# CNN Model for Text Classification

This notebook handles the entire pipeline for training a CNN model on text data:
1. Data loading and preprocessing
2. Text tokenization and sequence creation
3. Word embeddings
4. CNN model architecture
5. Model training and evaluation
6. Model saving


In [20]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.sequence import pad_sequences
import nltk
import os

nltk.download('punkt', quiet=True)

np.random.seed(2025)
tf.random.set_seed(2025)


## Data Preparation

In [21]:

print("Loading dataset...")
df = pd.read_csv("../datasets/custom_dataset.csv", sep="\t")
print(f"Dataset shape: {df.shape}")
df.head()


Loading dataset...
Dataset shape: (5437, 2)


,Text,Label
0,"In mechanics, a variable-mass system is a coll...",Human
1,Variable-mass systems in fluids involve object...,AI
2,"Geomechanics (from the Greek γεός, i.e. prefix...",Human
3,Geomechanics studies the mechanical behavior o...,AI
4,Microscale chemistry (often referred to as sma...,Human


In [22]:
import pickle

def predict_text(text, model, preprocessor):
    cleaned_text = preprocessor['clean_text'](text)
    
    sequence = preprocessor['tokenizer'].texts_to_sequences([cleaned_text])
    padded = pad_sequences(sequence, maxlen=preprocessor['max_seq_length'], padding='post', truncating='post')
    
    prediction = model.predict(padded)[0][0]
    
    return {
        'probability': float(prediction),
        'prediction': 'AI' if prediction > 0.5 else 'Human'
    }

loaded_model = keras.models.load_model('../trained_models/tensorflow/cnn_model.h5')
with open('../trained_models/tensorflow/cnn_tokenizer.pkl', 'rb') as f:
    loaded_preprocessor = pickle.load(f)

sample_text = "This is a sample text to test the model."
result = predict_text(sample_text, loaded_model, loaded_preprocessor)
print(f"Sample text: '{sample_text}'")
print(f"Prediction: {result['prediction']}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 301ms/step
Sample text: 'This is a sample text to test the model.'
Prediction: Human


## Load dataset

In [23]:
print("Loading dataset...")
try:
    df = pd.read_csv('../datasets/submission3_inputs.csv', sep=';')
except:
    df = pd.read_csv('../datasets/submission3_inputs.csv')

print(f"Dataset loaded with {len(df)} entries")

df.head()

Loading dataset...
Dataset loaded with 100 entries


,ID,Text
0,D3-1,String theory is a broad and varied subject th...
1,D3-2,String theory is a theoretical framework in ph...
2,D3-3,String theory proposes that the fundamental bu...
3,D3-4,I think string theory explains only the 3rd di...
4,D3-5,"With all this said, one should keep in mind th..."


## Make predictions with CNN Model

In [24]:

print("Making predictions with CNN model...")
cnn_predictions = []
for idx, row in df.iterrows():
    text = row['Text']
    prediction = predict_text(text, loaded_model, loaded_preprocessor)['prediction']
    cnn_predictions.append(prediction)
    if (idx + 1) % 10 == 0:
        print(f"Processed {idx + 1}/{len(df)} entries with CNN model")

cnn_results = pd.DataFrame({
    'ID': df['ID'],
    'Label': cnn_predictions
})

cnn_results.head()

Making predictions with CNN model...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
Processed 10/100 entries with CNN model
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
Processed 20/100 entries with CNN model
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━

,ID,Label
0,D3-1,Human
1,D3-2,AI
2,D3-3,AI
3,D3-4,Human
4,D3-5,Human


## Save Results

In [25]:

if not os.path.exists('results'):
    os.makedirs('results')

cnn_results.to_csv('results/submissao3-grupo011-s1.csv', sep='\t', index=False)